# exp410 Probability Threshold Optimization

**Context:** This is the source notebook for the threshold analysis
documented in Model Implementation 4.1.1.6 / Results 5.1.7
("Probability Threshold Optimization"). Converting a continuous peat
probability surface into a binary peat/non-peat mask requires choosing
a cutoff probability -- this notebook re-derives out-of-fold (OOF)
predictions from the 5 exp410 spatial-CV fold models and sweeps
candidate thresholds using three methods:

- **F1-optimal**: maximizes the harmonic mean of precision and recall
  -- this is the threshold that was ultimately SELECTED for production
  (0.327, referred to as ~0.33 elsewhere in the pipeline/scripts).
- **Youden's J**: maximizes sensitivity + specificity - 1 -- tested but
  NOT used, since at threshold 0.186 it over-predicted peat (~25% false
  positive rate).
- A 95th-percentile-recall candidate threshold is also computed for
  comparison.

The default/naive threshold of 0.50 is included as a baseline
comparison point -- documented as performing poorly (under-predicting
peat, missing areas visibly identifiable as peat in imagery).

The resulting F1-optimal threshold (0.327) is the same value used as
the peat-presence mask cutoff throughout the rest of the pipeline (depth,
composition, and other peat-property models restricted to pixels where
exp410 probability >= this threshold).


In [ ]:
# exp410 Threshold Optimization
# Finds the mathematically optimal probability cutoff for the peat mask
# using OOF predictions from exp410 cross-validation.
# Methods: Youden's J, F1-optimal, Precision-Recall curve
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

BASE       = '/scratch.global/ocon0444/peat_modeling'
MDL_DIR    = os.path.join(BASE, '03_models')
BINARY_CSV = os.path.join(BASE, '00_data/processed/binary_peat_features_0_20_dropped.csv')
EXP410_DIR = os.path.join(MDL_DIR, 'exp410')
OUT_DIR    = os.path.join(BASE, '05_results/threshold_optimization')
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL   = 'peat_binary'
RANDOM_STATE = 42
N_FOLDS      = 5
CURRENT_THRESH = 0.5

print('exp410 Threshold Optimization')

In [ ]:
# Cell 2 — Load exp410 models and regenerate OOF predictions
with open(os.path.join(EXP410_DIR, 'feature_list.json')) as f:
    feat_data = json.load(f)
feature_cols = feat_data['features']

models = [pickle.load(open(os.path.join(EXP410_DIR, f'model_fold_{i}.pkl'), 'rb'))
          for i in range(N_FOLDS)]

df = pd.read_csv(BINARY_CSV, low_memory=False)
if 'mn_nwi_binary' not in df.columns:
    df['mn_nwi_binary'] = (
        (df['mn_nwi_cowardin_1']==1)|(df['mn_nwi_cowardin_2']==1)
    ).astype(int)
if 'mn_nwi_merged_1_2' not in df.columns:
    df['mn_nwi_merged_1_2'] = df['mn_nwi_binary']

feat_cols = [c for c in feature_cols if c in df.columns]
X = df[feat_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
y = df[TARGET_COL].values

# Regenerate OOF predictions using same CV split
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_probs = np.zeros(len(y))

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    probs = models[fold].predict_proba(X.iloc[va_idx])[:, 1]
    oof_probs[va_idx] = probs

print(f'OOF predictions generated: {len(oof_probs):,} points')
print(f'Overall AUC  : {roc_auc_score(y, oof_probs):.4f}')
print(f'Overall AP   : {average_precision_score(y, oof_probs):.4f}')
print(f'Class balance: {y.mean():.3f} peat ({y.sum():,} peat / {(1-y).sum():,} non-peat)')

In [ ]:
# Cell 3 — Find optimal thresholds

# ROC curve -> Youden's J (maximizes sensitivity + specificity)
fpr, tpr, roc_thresholds = roc_curve(y, oof_probs)
specificity = 1 - fpr
youdens_j   = tpr + specificity - 1
best_j_idx  = np.argmax(youdens_j)
thresh_youden     = roc_thresholds[best_j_idx]
sens_youden       = tpr[best_j_idx]
spec_youden       = specificity[best_j_idx]

# Precision-Recall curve -> F1-optimal
precision, recall, pr_thresholds = precision_recall_curve(y, oof_probs)
f1_scores  = 2 * precision * recall / (precision + recall + 1e-9)
best_f1_idx = np.argmax(f1_scores)
thresh_f1   = pr_thresholds[min(best_f1_idx, len(pr_thresholds)-1)]
prec_f1     = precision[best_f1_idx]
rec_f1      = recall[best_f1_idx]
f1_best     = f1_scores[best_f1_idx]

# Evaluate a range of thresholds
thresholds  = np.arange(0.1, 0.9, 0.01)
thresh_stats = []
for t in thresholds:
    preds = (oof_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1   = f1_score(y, preds, zero_division=0)
    thresh_stats.append({
        'threshold': t, 'sensitivity': sens, 'specificity': spec,
        'precision': prec, 'f1': f1,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'peat_predicted_pct': preds.mean() * 100,
    })
stats_df = pd.DataFrame(thresh_stats)

print('OPTIMAL THRESHOLDS')
print('='*60)
print(f"Current (0.50)   : ", end='')
row = stats_df[stats_df['threshold'].round(2) == 0.50].iloc[0]
print(f"Sens={row['sensitivity']:.3f}  Spec={row['specificity']:.3f}  "
      f"F1={row['f1']:.3f}  {row['peat_predicted_pct']:.1f}% predicted peat")

print(f"Youden's J ({thresh_youden:.2f}) : "
      f"Sens={sens_youden:.3f}  Spec={spec_youden:.3f}")

print(f"F1-optimal ({thresh_f1:.2f}) : "
      f"Prec={prec_f1:.3f}  Rec={rec_f1:.3f}  F1={f1_best:.3f}")

# Also show threshold that captures 95% of peat (high recall)
high_recall = stats_df[stats_df['sensitivity'] >= 0.95]
if len(high_recall):
    hr_row = high_recall.iloc[-1]  # highest threshold that still hits 95% recall
    print(f"95% recall  ({hr_row['threshold']:.2f}) : "
          f"Sens={hr_row['sensitivity']:.3f}  Spec={hr_row['specificity']:.3f}  "
          f"F1={hr_row['f1']:.3f}  {hr_row['peat_predicted_pct']:.1f}% predicted peat")

In [ ]:
# Cell 4 — Diagnostic plots
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. ROC curve with thresholds marked
ax = axes[0, 0]
ax.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC={roc_auc_score(y, oof_probs):.4f})')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.scatter(1-spec_youden, sens_youden, color='red', s=120, zorder=5,
           label=f"Youden's J = {thresh_youden:.2f}")
# Mark current 0.5
row_05 = stats_df[stats_df['threshold'].round(2) == 0.50].iloc[0]
ax.scatter(1-row_05['specificity'], row_05['sensitivity'], color='orange',
           s=120, zorder=5, label=f'Current = 0.50', marker='D')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 2. Precision-Recall curve
ax = axes[0, 1]
ax.plot(recall, precision, 'g-', lw=2,
        label=f'PR (AP={average_precision_score(y, oof_probs):.4f})')
ax.scatter(rec_f1, prec_f1, color='red', s=120, zorder=5,
           label=f'F1-optimal = {thresh_f1:.2f}')
ax.scatter(row_05['sensitivity'], row_05['precision'], color='orange',
           s=120, zorder=5, label='Current = 0.50', marker='D')
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 3. F1 vs threshold
ax = axes[0, 2]
ax.plot(stats_df['threshold'], stats_df['f1'], 'b-', lw=2)
ax.axvline(thresh_f1, color='red', ls='--', lw=1.5, label=f'F1-optimal={thresh_f1:.2f}')
ax.axvline(CURRENT_THRESH, color='orange', ls='--', lw=1.5, label='Current=0.50')
ax.axvline(thresh_youden, color='green', ls='--', lw=1.5,
           label=f"Youden={thresh_youden:.2f}")
ax.set_xlabel('Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score vs Threshold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 4. Sensitivity & Specificity vs threshold
ax = axes[1, 0]
ax.plot(stats_df['threshold'], stats_df['sensitivity'], 'b-', lw=2, label='Sensitivity (recall)')
ax.plot(stats_df['threshold'], stats_df['specificity'], 'g-', lw=2, label='Specificity')
ax.plot(stats_df['threshold'], stats_df['precision'],   'r-', lw=2, label='Precision')
ax.axvline(thresh_youden, color='purple', ls='--', lw=1.5,
           label=f"Youden={thresh_youden:.2f} (sens=spec crossover)")
ax.axvline(CURRENT_THRESH, color='orange', ls='--', lw=1.5, label='Current=0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Sensitivity / Specificity / Precision vs Threshold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 5. % area predicted as peat vs threshold
ax = axes[1, 1]
ax.plot(stats_df['threshold'], stats_df['peat_predicted_pct'], 'b-', lw=2)
ax.axvline(thresh_youden, color='red', ls='--', lw=1.5, label=f"Youden={thresh_youden:.2f}")
ax.axvline(thresh_f1, color='green', ls='--', lw=1.5, label=f'F1-opt={thresh_f1:.2f}')
ax.axvline(CURRENT_THRESH, color='orange', ls='--', lw=1.5, label='Current=0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('% points predicted as peat')
ax.set_title('Predicted Peat Extent vs Threshold\n(proxy for spatial coverage)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 6. FP and FN rates vs threshold — tradeoff table
ax = axes[1, 2]
fn_rate = stats_df['fn'] / (stats_df['fn'] + stats_df['tp'])
fp_rate = stats_df['fp'] / (stats_df['fp'] + stats_df['tn'])
ax.plot(stats_df['threshold'], fn_rate * 100, 'r-', lw=2, label='Miss rate (FN%) — missing real peat')
ax.plot(stats_df['threshold'], fp_rate * 100, 'b-', lw=2, label='False alarm (FP%) — non-peat predicted peat')
ax.axvline(thresh_youden, color='purple', ls='--', lw=1.5, label=f"Youden={thresh_youden:.2f}")
ax.axvline(CURRENT_THRESH, color='orange', ls='--', lw=1.5, label='Current=0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('%')
ax.set_title('Miss Rate vs False Alarm Rate')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle(f'exp410 Threshold Optimization\n'
             f"Youden's J = {thresh_youden:.2f}  |  F1-optimal = {thresh_f1:.2f}  |  Current = {CURRENT_THRESH}",
             fontsize=13)
plt.tight_layout()
out = os.path.join(OUT_DIR, 'exp410_threshold_optimization.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Cell 5 — Summary table and save recommendation
candidates = {
    'current_0.50':  CURRENT_THRESH,
    'youden_j':      round(thresh_youden, 2),
    'f1_optimal':    round(thresh_f1, 2),
    '95pct_recall':  round(hr_row['threshold'], 2) if len(high_recall) else None,
}

print('THRESHOLD COMPARISON TABLE')
print('='*90)
print(f'{"Method":<18} {"Thresh":>7} {"Sensitivity":>12} {"Specificity":>12} '
      f'{"Precision":>10} {"F1":>7} {"Peat%":>7}')
print('-'*90)
for method, t in candidates.items():
    if t is None: continue
    t_round = round(t, 2)
    row = stats_df[(stats_df['threshold'] - t_round).abs() < 0.005].iloc[0]
    print(f"{method:<18} {t_round:>7.2f} {row['sensitivity']:>12.3f} "
          f"{row['specificity']:>12.3f} {row['precision']:>10.3f} "
          f"{row['f1']:>7.3f} {row['peat_predicted_pct']:>6.1f}%")
print('='*90)
print(f"\nRECOMMENDATION for depth mask:")
print(f"  If priority = don't miss peat   -> use 95% recall threshold ({candidates.get('95pct_recall','N/A')})")
print(f"  If priority = balanced           -> use Youden's J ({candidates['youden_j']})")
print(f"  If priority = best F1            -> use F1-optimal ({candidates['f1_optimal']})")

# Save
stats_df.to_csv(os.path.join(OUT_DIR, 'threshold_sweep.csv'), index=False)
json.dump({
    'exp_id': 'exp410',
    'current_threshold': CURRENT_THRESH,
    'youden_j_threshold': round(float(thresh_youden), 3),
    'f1_optimal_threshold': round(float(thresh_f1), 3),
    '95pct_recall_threshold': round(float(hr_row['threshold']), 3) if len(high_recall) else None,
    'auc': round(roc_auc_score(y, oof_probs), 4),
    'ap': round(average_precision_score(y, oof_probs), 4),
}, open(os.path.join(OUT_DIR, 'optimal_thresholds.json'), 'w'), indent=2)
print(f'\nResults saved to {OUT_DIR}')